# Iris Classification with Neural Networks

In [ ]:
import pandas as pd

from keras import layers, backend, Sequential
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

from functools import partial

## Dataset

In [ ]:
df = pd.read_csv("datasets/iris.csv")
display(df.head())
display(df.describe())

### Encode

In [ ]:
df_encoded = pd.get_dummies(df, columns=["class"], prefix="class", dtype=int)
display(df_encoded.sample(5))

### Feature/Target Split

In [ ]:
X = df_encoded[["sepal length", "sepal width", "petal length", "petal width"]]
y = df_encoded[["class_Iris-setosa", "class_Iris-versicolor", "class_Iris-virginica"]]

display(X)
display(y)

## Model

### Definition

In [ ]:
def get_neural_network(
    hidden_layers:int,
    activation:str,
    kernel_initializer:str,
    optimizer:str,
) -> Sequential:
    backend.clear_session()


    model = Sequential()
    model.add(layers.InputLayer(shape=(4,)))

    for _ in range(hidden_layers):
        model.add(layers.Dense(
            units=4, 
            activation=activation,
            kernel_initializer=kernel_initializer
        ))

    model.add(layers.Dense(units=3, activation="softmax"))


    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=['categorical_accuracy']
    )
    
    return model

model = KerasClassifier(model=get_neural_network, verbose=0)

### Tuning

In [ ]:
params = {
    'batch_size': [10, 20],
    'epochs': [50, 100],
    'model__hidden_layers': [0, 1, 2],
    'model__activation': ['relu', 'tanh'],
    'model__kernel_initializer': ['normal', 'random_uniform'],
    'model__optimizer': ['adam', 'sgd']
}

tuning_results = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring='accuracy',
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=2
).fit(X, y) # 32 min

In [ ]:
print(tuning_results.best_params_) # {'batch_size': 10, 'epochs': 100, 'model__activation': 'tanh', 'model__hidden_layers': 1, 'model__kernel_initializer': 'normal', 'model__optimizer': 'adam'}
print(tuning_results.best_score_) # 0.9800000000000001

### Best Model Evaluation

In [ ]:
best_model = KerasClassifier(
    model = partial(
        get_neural_network,
        hidden_layers=1,
        activation='tanh',
        kernel_initializer='normal',
        optimizer='adam',
    ),
    epochs=100,
    batch_size=10,
    verbose=0
)

results = cross_val_score(
    estimator=best_model,
    X=X, y=y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
)

In [ ]:
print(f"Accuracies: {results}")
print(f"Mean: {results.mean()}, Std: {results.std()}")